# Register HNN SONATA circuits on staging

Registers the three circuits produced by `scripts/hnn_to_sonata.py` (`jones_2009`, `law_2021`, `calcium`) as `Circuit` entities in the OBI staging deployment and uploads each folder as a `sonata_circuit` asset.

**Target**
- Environment: `staging`
- Virtual lab: `e6030ed8-a589-4be2-80a6-f975406eb1f6`
- Project:     `2720f785-a3a2-4472-969d-19a53891c817`

Follows the pattern from `obi_one/scientific/tasks/em_synapse_mapping/register.py` and `obi_one/utils/db_sdk.py::add_circuit_folder_asset`.

In [ ]:
from pathlib import Path

import h5py
import numpy as np
from entitysdk import Client, ProjectContext, models, types
from entitysdk._server_schemas import AssetLabel, CircuitBuildCategory, CircuitScale
from obi_auth import get_token

In [ ]:
# ---- configuration ---------------------------------------------------------
ENVIRONMENT = "staging"
VLAB_ID = "e6030ed8-a589-4be2-80a6-f975406eb1f6"
PROJECT_ID = "2720f785-a3a2-4472-969d-19a53891c817"

# repo root is one level above this notebook
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
CIRCUITS_ROOT = REPO_ROOT / "sonata_circuits"
assert CIRCUITS_ROOT.is_dir(), f"Expected circuits at {CIRCUITS_ROOT}"

# Human somatosensory cortex — HNN models are fit to human MEG/EEG signals.
SPECIES_NAME = "Homo sapiens"
BRAIN_REGION_ACRONYM = "SSp"  # Primary somatosensory area (Allen CCF)

MODELS = {
    "jones_2009": {
        "name": "HNN Jones et al. 2009 default model (SONATA)",
        "description": (
            "SONATA conversion of the HNN `jones_2009_model` from hnn-core. "
            "Laminar cortical column (L2/3 + L5 pyramidals and basket cells) with "
            "distance-dependent AMPA/NMDA/GABA_A/GABA_B connectivity. Soma positions "
            "are placed on a 10x10 in-plane mesh following the hnn-core default. "
            "Reference: Jones SR et al., J. Neurophysiol. 2009."
        ),
    },
    "law_2021": {
        "name": "HNN Law et al. 2021 model (SONATA)",
        "description": (
            "SONATA conversion of the HNN `law_2021_model` from hnn-core. Extends "
            "the Jones 2009 network with Law et al. 2021 modifications: L5-pyr NMDA "
            "weight reduced, tau1=45ms/tau2=200ms on L5+L2 pyramidal GABA_B, new "
            "L5-basket to L5-pyr distal GABA_A, L2-basket to L5-pyr connection "
            "replaced by GABA_B, and ca mechanism removed from L5-pyr soma/basal."
        ),
    },
    "calcium": {
        "name": "HNN calcium model (Kohl et al. 2022) (SONATA)",
        "description": (
            "SONATA conversion of the HNN `calcium_model` from hnn-core. Jones 2009 "
            "network with distance-dependent calcium channel density on pyramidal "
            "dendrites as used in Kohl et al., Cerebral Cortex 2022."
        ),
    },
}

In [ ]:
# ---- authenticate & build client ------------------------------------------
token = get_token(environment=ENVIRONMENT)
project_context = ProjectContext(
    virtual_lab_id=VLAB_ID,
    project_id=PROJECT_ID,
    environment=ENVIRONMENT,
)
client = Client(
    environment=ENVIRONMENT,
    project_context=project_context,
    token_manager=token,
)
print(f"Client ready: {client.api_url}")

In [ ]:
# ---- resolve Species and BrainRegion --------------------------------------
species_hits = list(
    client.search_entity(entity_type=models.Species, query={"name": SPECIES_NAME}).all()
)
assert species_hits, f"Species '{SPECIES_NAME}' not found on {ENVIRONMENT}"
species = species_hits[0]

region_hits = list(
    client.search_entity(
        entity_type=models.BrainRegion, query={"acronym": BRAIN_REGION_ACRONYM}
    ).all()
)
assert region_hits, f"Brain region '{BRAIN_REGION_ACRONYM}' not found on {ENVIRONMENT}"
brain_region = region_hits[0]

print(f"Species:      {species.name}  ({species.id})")
print(f"Brain region: {brain_region.name} ({brain_region.acronym}) ({brain_region.id})")

In [ ]:
# ---- subject: reuse existing 'HNN human' subject or create one ------------
SUBJECT_NAME = "HNN human generic subject"
existing = list(
    client.search_entity(entity_type=models.Subject, query={"name": SUBJECT_NAME}).all()
)
if existing:
    subject = existing[0]
    print(f"Reusing Subject {subject.id}")
else:
    subject = client.register_entity(
        models.Subject(
            name=SUBJECT_NAME,
            description="Generic human subject used to host HNN cortical circuit models.",
            species=species,
            sex=types.Sex.unknown,
            age_period=types.AgePeriod.postnatal,
        )
    )
    print(f"Created Subject {subject.id}")

In [ ]:
# ---- helpers ---------------------------------------------------------------
def circuit_counts(circuit_dir: Path) -> tuple[int, int, int]:
    """Count nodes, edges, and unique (pre,post) pairs across all populations."""
    n_nodes = 0
    for nodes_file in circuit_dir.rglob("nodes.h5"):
        with h5py.File(nodes_file) as f:
            for pop in f["nodes"]:
                n_nodes += int(f[f"nodes/{pop}/node_type_id"].shape[0])

    n_edges = 0
    pairs: set[tuple[int, int]] = set()
    for edges_file in circuit_dir.rglob("edges.h5"):
        with h5py.File(edges_file) as f:
            for pop in f["edges"]:
                n_edges += int(f[f"edges/{pop}/edge_type_id"].shape[0])
                src = f[f"edges/{pop}/source_node_id"][:]
                tgt = f[f"edges/{pop}/target_node_id"][:]
                pairs.update(zip(src.tolist(), tgt.tolist()))
    return n_nodes, n_edges, len(pairs)


def collect_circuit_files(circuit_dir: Path) -> dict:
    """Map of POSIX relative path -> absolute path for every file in the folder."""
    files = {
        str(p.relative_to(circuit_dir)): p
        for p in circuit_dir.rglob("*")
        if p.is_file()
    }
    assert "circuit_config.json" in files, f"circuit_config.json missing in {circuit_dir}"
    assert "node_sets.json" in files, f"node_sets.json missing in {circuit_dir}"
    return files


for slug in MODELS:
    n, e, c = circuit_counts(CIRCUITS_ROOT / slug)
    print(f"{slug:<10}  neurons={n:>4}  synapses={e:>6}  connections={c:>6}")

In [ ]:
# ---- register & upload each circuit ---------------------------------------
# Safety: skip re-registering a circuit that already exists in this project.
registered: dict[str, models.Circuit] = {}

for slug, meta in MODELS.items():
    circuit_dir = CIRCUITS_ROOT / slug
    name = meta["name"]
    description = meta["description"]

    existing = list(
        client.search_entity(entity_type=models.Circuit, query={"name": name}).all()
    )
    if existing:
        print(f"[skip] '{name}' already registered: {existing[0].id}")
        registered[slug] = existing[0]
        continue

    n_neurons, n_synapses, n_connections = circuit_counts(circuit_dir)

    circuit_model = models.Circuit(
        name=name,
        description=description,
        subject=subject,
        brain_region=brain_region,
        number_neurons=n_neurons,
        number_synapses=n_synapses,
        number_connections=n_connections,
        scale=CircuitScale.microcircuit,
        build_category=CircuitBuildCategory.computational_model,
        has_morphologies=True,
        has_point_neurons=False,
        has_electrical_cell_models=True,
        has_spines=False,
        contact_email="isbisterjb@gmail.com",
        published_in="Jones SR et al., J. Neurophysiol. 102:3554 (2009)",
        authorized_public=False,
    )
    circuit_entity = client.register_entity(circuit_model)
    print(f"[ok]   registered Circuit '{name}' -> {circuit_entity.id}")

    paths = collect_circuit_files(circuit_dir)
    asset = client.upload_directory(
        entity_id=circuit_entity.id,
        entity_type=models.Circuit,
        name="sonata_circuit",
        paths=paths,
        label=AssetLabel.sonata_circuit,
    )
    print(f"       uploaded {len(paths)} files as asset {asset.id}")
    registered[slug] = circuit_entity

In [ ]:
# ---- summary ---------------------------------------------------------------
for slug, ent in registered.items():
    print(f"{slug:<10}  {ent.id}  {ent.name}")

In [ ]:
# ---- patch: fix scale on already-registered circuits ----------------------
# Earlier registrations used CircuitScale.small; these models belong to
# CircuitScale.microcircuit. Re-run safely — it's idempotent.
for slug, meta in MODELS.items():
    hits = list(
        client.search_entity(entity_type=models.Circuit, query={"name": meta["name"]}).all()
    )
    assert hits, f"Circuit '{meta['name']}' not found — register it first."
    ent = hits[0]
    updated = client.update_entity(
        entity_id=ent.id,
        entity_type=models.Circuit,
        attrs_or_entity={"scale": CircuitScale.microcircuit},
    )
    print(f"{slug:<10}  {ent.id}  scale -> {updated.scale}")

In [ ]:
# ---- patch: replace sonata_circuit asset with freshly-rendered hoc files --
# After re-rendering `emodels_hoc/*.hoc` from `cell_templates/*.json`
# (via `python scripts/hnn_to_sonata.py --regen-hocs`), the directory
# asset on staging still holds the old stub hocs. Re-upload the whole
# sonata_circuit directory: entitysdk has no partial-update for dir assets,
# so we delete + re-upload. Not atomic — if upload fails, re-run the cell.
for slug, meta in MODELS.items():
    circuit_dir = CIRCUITS_ROOT / slug
    hits = list(
        client.search_entity(entity_type=models.Circuit, query={"name": meta["name"]}).all()
    )
    assert hits, f"Circuit '{meta['name']}' not found — register it first."
    ent = hits[0]

    assets = list(client.get_entity_assets(entity_id=ent.id, entity_type=models.Circuit).all())
    dir_assets = [a for a in assets if a.label == AssetLabel.sonata_circuit and a.is_directory]
    for old in dir_assets:
        client.delete_asset(entity_id=ent.id, entity_type=models.Circuit, asset_id=old.id)
        print(f"  deleted old asset {old.id}")

    paths = collect_circuit_files(circuit_dir)
    asset = client.upload_directory(
        entity_id=ent.id,
        entity_type=models.Circuit,
        name="sonata_circuit",
        paths=paths,
        label=AssetLabel.sonata_circuit,
    )
    print(f"{slug:<10}  {ent.id}  re-uploaded {len(paths)} files -> asset {asset.id}")

In [ ]:
# ---- upload circuit_connectivity_matrices asset ---------------------------
# Each circuit directory contains a `connectivity_matrices/` folder with a
# `matrix_config.json` + `<edge_pop>/single/connectivity_matrix.h5` produced
# by `conntility.ConnectivityMatrix.to_h5(...)` (same schema as the
# `circuit_connectivity_matrices` asset on public OBI circuits). Upload it as
# a directory asset with that label. Idempotent: delete any pre-existing
# `circuit_connectivity_matrices` asset on the entity before re-uploading.
for slug, meta in MODELS.items():
    matrix_dir = CIRCUITS_ROOT / slug / "connectivity_matrices"
    assert matrix_dir.is_dir(), f"Missing {matrix_dir} — run the matrix builder first."

    hits = list(
        client.search_entity(entity_type=models.Circuit, query={"name": meta["name"]}).all()
    )
    assert hits, f"Circuit '{meta['name']}' not found — register it first."
    ent = hits[0]

    assets = list(client.get_entity_assets(entity_id=ent.id, entity_type=models.Circuit).all())
    old_matrix_assets = [
        a for a in assets
        if a.label == AssetLabel.circuit_connectivity_matrices and a.is_directory
    ]
    for old in old_matrix_assets:
        client.delete_asset(entity_id=ent.id, entity_type=models.Circuit, asset_id=old.id)
        print(f"  deleted old connectivity-matrix asset {old.id}")

    matrix_files = {
        str(p.relative_to(matrix_dir)): p
        for p in matrix_dir.rglob("*")
        if p.is_file()
    }
    asset = client.upload_directory(
        entity_id=ent.id,
        entity_type=models.Circuit,
        name="circuit_connectivity_matrices",
        paths=matrix_files,
        label=AssetLabel.circuit_connectivity_matrices,
    )
    print(f"{slug:<10}  {ent.id}  uploaded {len(matrix_files)} files -> asset {asset.id}")